# 01 — Data extraction

**Goal**: produce two benchmark datasets for the 4-category classifier.

Benchmarks:
1. `rule_based`: records that already have a rule label in `spam`, `noise`, `service_feedback`, `config_feedback`, or `app_specific`; `spam` + `noise` are merged into `junk`.
2. `hand_labelled`: records from the rule-based `others` bucket that were manually labelled in `scripts/labelled/others_gold_v1.csv`.

Both benchmarks use only the four real classifier labels: `junk`, `service_feedback`, `config_feedback`, and `app_specific`. `others` is only the rule-based source bucket and is not a model output label.

Outputs:
- `data/splits/rule_based/{train,val,test}.parquet`
- `data/splits/hand_labelled/test.parquet`
- Backwards-compatible aliases: `data/splits/{train,val,test}.parquet` point to the `rule_based` split.


In [1]:
import sys
from pathlib import Path
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(AI_ROOT))

import pandas as pd
from shared.data_loader import load_hand_labelled_csv, split_train_val_test, stratified_sample
from shared.types import LLM_OUTPUT_CATEGORIES

BACKEND_ROOT = AI_ROOT.parent
GOLD_CSV = BACKEND_ROOT / 'scripts' / 'labelled' / 'others_gold_v1.csv'
RULE_SOURCE_CATEGORIES = ['spam', 'noise', 'service_feedback', 'config_feedback', 'app_specific']
print('gold csv:', GOLD_CSV, '(exists =', GOLD_CSV.exists(), ')')
print('llm labels:', LLM_OUTPUT_CATEGORIES)

gold csv: /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/scripts/labelled/others_gold_v1.csv (exists = True )
llm labels: ['junk', 'service_feedback', 'config_feedback', 'app_specific']


In [2]:
PER_CAT = 2000
rule_based = stratified_sample(per_category=PER_CAT, seed=42, categories=RULE_SOURCE_CATEGORIES)
rule_based = rule_based.rename(columns={'rule_category': 'label'})
rule_based = rule_based[rule_based['label'].isin(LLM_OUTPUT_CATEGORIES)].copy()
rule_based['bench'] = 'rule_based'
print('rule_based rows:', len(rule_based))
rule_based['label'].value_counts()

rule_based rows: 6815


label
service_feedback    2000
config_feedback     2000
app_specific        2000
junk                 815
Name: count, dtype: int64

In [3]:
hand_labelled = load_hand_labelled_csv(GOLD_CSV)
print('hand_labelled rows:', len(hand_labelled))
hand_labelled['label'].value_counts()

hand_labelled rows: 197


label
service_feedback    138
app_specific         36
config_feedback      22
junk                  1
Name: count, dtype: int64

In [4]:
rule_train, rule_val, rule_test = split_train_val_test(
    rule_based, label_col='label', train_frac=0.7, val_frac=0.15, seed=42,
)

bench_splits = {
    'rule_based': {
        'train': rule_train,
        'val': rule_val,
        'test': rule_test,
    },
    'hand_labelled': {
        'test': hand_labelled,
    },
}

for bench, splits in bench_splits.items():
    print(f'[{bench}]')
    for name, part in splits.items():
        print(f'  {name:5s} {len(part):>5}  ', dict(part['label'].value_counts()))

[rule_based]
  train  4770   {'app_specific': np.int64(1400), 'config_feedback': np.int64(1400), 'service_feedback': np.int64(1400), 'junk': np.int64(570)}
  val    1022   {'service_feedback': np.int64(300), 'config_feedback': np.int64(300), 'app_specific': np.int64(300), 'junk': np.int64(122)}
  test   1023   {'config_feedback': np.int64(300), 'app_specific': np.int64(300), 'service_feedback': np.int64(300), 'junk': np.int64(123)}
[hand_labelled]
  test    197   {'service_feedback': np.int64(138), 'app_specific': np.int64(36), 'config_feedback': np.int64(22), 'junk': np.int64(1)}


In [5]:
SPLITS_DIR = AI_ROOT / 'data' / 'splits'
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# `feedback_parsed` is dict-typed; serialise to JSON string for Parquet safety.
import json

def parquet_safe(part: pd.DataFrame) -> pd.DataFrame:
    part = part.copy()
    part['feedback_parsed'] = part['feedback_parsed'].apply(
        lambda v: json.dumps(v, ensure_ascii=False) if isinstance(v, dict) else (v or '')
    )
    return part

for bench, splits in bench_splits.items():
    bench_dir = SPLITS_DIR / bench
    bench_dir.mkdir(parents=True, exist_ok=True)
    for name, part in splits.items():
        safe = parquet_safe(part)
        out = bench_dir / f'{name}.parquet'
        safe.to_parquet(out, index=False)
        print('wrote', out, len(safe))

        # Keep legacy paths usable for notebooks/scripts that expect the original split location.
        if bench == 'rule_based':
            legacy_out = SPLITS_DIR / f'{name}.parquet'
            safe.to_parquet(legacy_out, index=False)
            print('wrote alias', legacy_out, len(safe))

wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/rule_based/train.parquet 4770
wrote alias /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/train.parquet 4770
wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/rule_based/val.parquet 1022
wrote alias /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/val.parquet 1022
wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/rule_based/test.parquet 1023
wrote alias /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/test.parquet 1023
wrote /Users/macbook/Documents/Code/Bachelor_Thesis/erc-8004-benchmarking-be/AI/data/splits/hand_labelled/test.parquet 197
